# Phase 4 (LSTM) — MIL ablation rerun with LSTM encoder

Reruns the three attention-pooling variants from `phase4_mil` with an LSTM
encoder instead of GRU:

- **MIL_resolved**: phase-resolved attention (per-phase scoring MLP)
- **MIL_shared**: phase-shared attention (one scoring MLP)
- **mean_pool**: forced uniform attention weights

All run on the same fold (All streams, 2020 LOYO test year) so the
comparison is apples-to-apples with the previous GRU MIL results.

**Approach:** monkey-patch `phase4_mil.model.StreamEncoder` to use `nn.LSTM`
instead of `nn.GRU` before instantiating any models. Everything else
(BagAttention, MILModel, training loop, evaluation, attention weight
collection) is reused unchanged from the existing package.

**Runtime estimate:** ~15 to 25 minutes total (three variants × one fold).

Output: `phase4_outputs_lstm/mil_lstm_predictions.npz` and a per-phase
comparison table.


## 1. Setup + monkey-patch the encoder

In [1]:
from pathlib import Path
import sys, os, time, gc
PROJECT_ROOT = Path('/fs1/home/atikul/Phenology')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from torch import nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


Device: cuda


In [2]:
# Monkey-patch StreamEncoder BEFORE any other phase4_mil imports
import phase4_mil.model as _mil_model


class StreamEncoderLSTM(nn.Module):
    """LSTM-based replacement for phase4_mil.model.StreamEncoder.

    Identical interface (forward takes (x, m), preserves leading bag dims,
    returns hidden states of shape (..., T, 2H)). Internally uses nn.LSTM
    instead of nn.GRU.
    """
    def __init__(self, in_dim, hidden, num_layers=1, dropout=0.0):
        super().__init__()
        self.rnn = nn.LSTM(input_size=2 * in_dim, hidden_size=hidden,
                            num_layers=num_layers, batch_first=True,
                            bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.out_dim = 2 * hidden

    def forward(self, x, m):
        lead = x.shape[:-2]
        T = x.shape[-2]; F = x.shape[-1]
        x_flat = x.reshape(-1, T, F)
        m_flat = m.reshape(-1, T, F)
        inp = torch.cat([x_flat * m_flat, m_flat], dim=-1)
        h, _ = self.rnn(inp)                   # LSTM returns (h, (h_n, c_n)); only need h
        return h.reshape(*lead, T, self.out_dim)


# Replace the GRU encoder in the module
_mil_model.StreamEncoder = StreamEncoderLSTM
print('Monkey-patched phase4_mil.model.StreamEncoder -> LSTM variant')

# Now import the rest of the package (it will pick up the LSTM encoder
# through phase4_mil.model.StreamEncoder when MILModel constructs streams)
from phase4_mil.bag_dataset import build_bags
from phase4_mil.train import train_mil_fold

# Verify the patch worked: MILModel.streams should now contain LSTM encoders
from phase4_mil.model import MILModel, StreamEncoder as PatchedEncoder
print('phase4_mil.model.StreamEncoder is LSTM variant:',
       hasattr(PatchedEncoder, '__init__') and 'lstm' in str(PatchedEncoder.__init__).lower() or
       PatchedEncoder.__name__ == 'StreamEncoderLSTM')


Monkey-patched phase4_mil.model.StreamEncoder -> LSTM variant
phase4_mil.model.StreamEncoder is LSTM variant: True


## 2. Load data + build bags for the 2020 LOYO fold

In [3]:
from phase3_baselines.dataset import (load_year_bundle, SelectionPolicy,
                                       apply_selection, concat_bundles)

TENSORS_DIR = PROJECT_ROOT / 'cache' / 'tensors'
YEARS = [2017, 2018, 2019, 2020, 2021]
TEST_YEAR = 2020

policy = SelectionPolicy(use_only_rank1=False)  # MIL uses all candidate fields per bag

print('Loading bundles...')
test_b = apply_selection(
    load_year_bundle(TENSORS_DIR / f'phenology_{TEST_YEAR}.npz'), policy)
train_parts = {}
for y in YEARS:
    if y == TEST_YEAR: continue
    train_parts[y] = apply_selection(
        load_year_bundle(TENSORS_DIR / f'phenology_{y}.npz'), policy)
train_b = concat_bundles(train_parts, years=[y for y in YEARS if y != TEST_YEAR])
del train_parts; gc.collect()
print(f'  train records: {train_b["X"].shape[0]}, test records: {test_b["X"].shape[0]}')

# Group records into bags (10 fields per station)
print('Building bags...')
train_bags = build_bags(train_b)
test_bags  = build_bags(test_b)
print(f'  train bags: {len(train_bags["bag_ids"])}, test bags: {len(test_bags["bag_ids"])}')


Loading bundles...
  train records: 21634, test records: 5181
Building bags...


KeyError: 'bag_ids'

## 3. Train each MIL variant

Each variant is one full training of the All-streams MIL model on the 2020
fold. Hyperparameters match the previous GRU MIL run, with hidden bumped to
64 to be consistent with the LSTM baseline grid.


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'phase4_outputs_lstm'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# === Hyperparameters (match the LSTM full grid for consistency) ===
HIDDEN     = 64
EPOCHS     = 12
BATCH_SIZE = 8        # bag-level batch (8 stations × 10 fields = 80 records per batch)
LR         = 1e-3
WD         = 1e-5
DROPOUT    = 0.1
ACTIVE     = ('S1', 'S2', 'Met')
FIELD_LOSS_WEIGHT = 0.3

print(f'Config: hidden={HIDDEN}, epochs={EPOCHS}, batch_size={BATCH_SIZE}')
print(f'Active streams: {ACTIVE}')


In [ ]:
# Phase-resolved attention
print('\n=== Training MIL_resolved (phase-resolved attention, LSTM) ===')
t0 = time.time()
result_resolved = train_mil_fold(
    train_bags=train_bags, test_bags=test_bags,
    active=ACTIVE,
    hidden=HIDDEN, num_layers=1, epochs=EPOCHS, batch_size=BATCH_SIZE,
    lr=LR, weight_decay=WD, dropout=DROPOUT,
    phase_resolved_attention=True,
    field_loss_weight=FIELD_LOSS_WEIGHT,
    use_amp=True, device=DEVICE, verbose=True,
)
print(f'MIL_resolved done in {time.time()-t0:.0f}s')
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# Phase-shared attention
print('\n=== Training MIL_shared (phase-shared attention, LSTM) ===')
t0 = time.time()
result_shared = train_mil_fold(
    train_bags=train_bags, test_bags=test_bags,
    active=ACTIVE,
    hidden=HIDDEN, num_layers=1, epochs=EPOCHS, batch_size=BATCH_SIZE,
    lr=LR, weight_decay=WD, dropout=DROPOUT,
    phase_resolved_attention=False,   # KEY difference
    field_loss_weight=FIELD_LOSS_WEIGHT,
    use_amp=True, device=DEVICE, verbose=True,
)
print(f'MIL_shared done in {time.time()-t0:.0f}s')
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# Mean pool (no attention): we achieve this by forcing the attention output
# to uniform weights. The simplest path is to use the phase4_mil training but
# replace the attention forward with uniform pooling, AFTER the model is built.
# Approach: train normally with phase_resolved_attention=False, then at
# inference time replace alpha with uniform weights. But since we need the
# *trained* model to use uniform pooling during training (so the field reps
# don't learn to rely on attention), the cleanest implementation is to
# monkey-patch BagAttention to use uniform weights.

import phase4_mil.model as _mil_model_v2

class BagAttentionUniform(nn.Module):
    """Forces uniform 1/B attention weights across valid slots.

    Same interface as BagAttention. Forward returns (pooled, alpha) where
    alpha is uniform over valid slots and zero elsewhere.
    """
    def __init__(self, hidden, n_phases, score_hidden=32,
                  phase_resolved=False, dropout=0.1):
        super().__init__()
        self.hidden = hidden
        self.n_phases = n_phases

    def forward(self, h_field, h_seq, valid_mask):
        # h_field: (B_bags, B, hidden)
        # valid_mask: (B_bags, B) bool
        B_bags, B, _ = h_field.shape
        valid_f = valid_mask.float()                       # (B_bags, B)
        counts = valid_f.sum(dim=1, keepdim=True).clamp(min=1.0)   # (B_bags, 1)
        alpha_base = valid_f / counts                     # (B_bags, B)
        # Broadcast to phase dimension: (B_bags, B, P) all phases share the same uniform alpha
        alpha = alpha_base.unsqueeze(-1).expand(B_bags, B, self.n_phases)
        # Pool h_field per phase: each phase gets the same pooled rep (uniform pool)
        # pooled shape: (B_bags, P, hidden)
        pooled = (h_field.unsqueeze(2) * alpha.unsqueeze(-1)).sum(dim=1)
        return pooled, alpha

# Save the real attention class so we can restore it afterwards
_real_attention = _mil_model_v2.BagAttention
_mil_model_v2.BagAttention = BagAttentionUniform

print('\n=== Training mean_pool (uniform attention, LSTM) ===')
t0 = time.time()
try:
    result_meanpool = train_mil_fold(
        train_bags=train_bags, test_bags=test_bags,
        active=ACTIVE,
        hidden=HIDDEN, num_layers=1, epochs=EPOCHS, batch_size=BATCH_SIZE,
        lr=LR, weight_decay=WD, dropout=DROPOUT,
        phase_resolved_attention=False,   # ignored by uniform attention
        field_loss_weight=FIELD_LOSS_WEIGHT,
        use_amp=True, device=DEVICE, verbose=True,
    )
finally:
    # Always restore the real attention class
    _mil_model_v2.BagAttention = _real_attention
print(f'mean_pool done in {time.time()-t0:.0f}s')
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


## 4. Side-by-side comparison

In [ ]:
def summarize(result, name):
    """Compute per-phase MAE for the station-level predictions."""
    from phase3_baselines.evaluate import evaluate_predictions
    rep = evaluate_predictions(
        pred_days=result['station_pred'].astype(np.float32),
        true_days=result['station_truth'].astype(np.float32),
        phase_ids=train_b['phase_ids'],
        years=result['station_year'],
        ranks=np.ones_like(result['station_year']),   # station-level: rank doesn't apply
    )
    out = rep['per_phase'][['mae', 'within_5d']].copy()
    out.columns = [f'{name}_{c}' for c in out.columns]
    macro = rep['headline']['macro_mae']
    return out, macro

table_resolved, macro_resolved = summarize(result_resolved, 'MIL_resolved')
table_shared,   macro_shared   = summarize(result_shared,   'MIL_shared')
table_meanpool, macro_meanpool = summarize(result_meanpool, 'mean_pool')

combined = table_resolved.join(table_shared).join(table_meanpool)
print('=== Per-phase station-level MAE (2020 fold, All streams, LSTM encoder) ===')
print(combined.round(2).to_string())
print()
print('=== Macro MAE ===')
print(f'  MIL_resolved: {macro_resolved:.3f} days')
print(f'  MIL_shared:   {macro_shared:.3f} days')
print(f'  mean_pool:    {macro_meanpool:.3f} days')

combined.to_csv(OUTPUT_DIR / 'mil_lstm_per_phase_2020.csv')


## 5. Save attention weights for figure F5

In [ ]:
# F5 needs the mean attention weights per (slot, phase) for each variant.
# train_mil_fold returns attention weights as `alpha` in the result dict
# (shape: N_bags × max_slots × n_phases, or similar).

import numpy as np

# Save in the same format as the original mil_partA_predictions.npz so figure
# F5 picks it up automatically.
LOGITS_NPZ = OUTPUT_DIR / 'mil_partA_predictions_lstm.npz'

# Construct the keys expected by fig5_attention_heatmap:
#   {cond}__{some_key}__alpha   (the code only looks for the prefix)
arrays = {}
for cond, result in [('MIL_resolved', result_resolved),
                      ('MIL_shared',   result_shared),
                      ('mean_pool',    result_meanpool)]:
    alpha = result.get('alpha')
    if alpha is None:
        print(f'WARN: no alpha in {cond}; skipping')
        continue
    arrays[f'{cond}__2020__alpha']    = np.asarray(alpha, dtype=np.float32)
    arrays[f'{cond}__2020__bag_id']   = np.asarray(result['station_bag'], dtype=np.int64)

np.savez_compressed(LOGITS_NPZ, **arrays)
print(f'Saved attention weights -> {LOGITS_NPZ}')
print(f'Keys: {list(arrays.keys())}')


---

## What to paste back to me

The two tables from Section 4:

1. The **per-phase MAE comparison** for the three MIL variants
2. The **macro MAE summary**

If the macro MAE pattern is similar to the previous GRU MIL run (all three
variants within 0.5 days of each other), the manuscript's negative-result
framing of Section 4.5 stays unchanged — only the architecture name changes
from GRU to LSTM. If the pattern is materially different, we'll need to
revisit Section 4.5's conclusion.
